In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
from torchvision.transforms.v2.functional import to_pil_image

%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import torch.nn.functional as F
from common import *

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors_darker = sns.color_palette("dark", 5 * 2)
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
import os

def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)


def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
ratios = [0.05, 0.075, 0.1, 0.2]
# ratios = [0.03, 0.05, 0.07, 0.09]
lambdas = 0.05
# lambdas = 0.01
# ratios = [0.2]
# lambdas = 0.01
loss = []
acc_object = []
acc_background = []
steps = []
eig_list = []
method = "BarlowTwins_single_background"
# method = "BarlowTwins_multiple_filtered"
directory_models = f"/models/jhkim/waterbirds/{method}_{lambdas}"
directory = f"datas/{method}_{lambdas}"
print(directory)
# directory = f"datas/lambda_{method}_{lambdas}"
for r in ratios:
    loss_array = load_from_csv(f"{directory}/{r}/loss.csv")
    [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{r}/accuracy.csv")
    eigvals_lists_array = load_from_csv_split(f"{directory}/{r}/eig.csv")
    loss.append(loss_array)
    acc_object.append(acc_object_array)
    acc_background.append(acc_background_array)
    steps.append(step_array)
    eig_list.append(eigvals_lists_array)

In [ ]:
background_predictability = 70

fig, axs = plt.subplots(1, 2, figsize=(28, 10))

# plot_range = 300

positions = []
plateau_range = 3.5
for r in range(len(ratios)):
    pos = np.where(acc_object[r] > background_predictability + plateau_range)
    if len(pos[0]) > 0:
        positions.append(pos[0][0])
        axs[0].axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
        axs[1].axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
    axs[0].plot(steps[r], smooth_data(acc_object[r]), alpha=0.6, linewidth=5, color=colors_darker[r],
                label=f"Object ratio : {ratios[r]:.3f}")
    axs[0].plot(steps[r], smooth_data(acc_background[r]), linestyle=":", alpha=0.75, linewidth=4, color=colors[r], label=f"Object ratio : {ratios[r]:.3f}")

upper_y = np.zeros_like(steps[0]) + background_predictability + plateau_range
lower_y = np.zeros_like(steps[0]) + background_predictability
axs[0].fill_between(steps[0], upper_y, lower_y, color='skyblue', alpha=0.4)
# axs[0].text(500, 67, "bg correlation")

# axs.set_xlim(0-1, epochs)
# axs.set_ylim(10, 100)
axs[0].axhline(background_predictability, linestyle="--", color="black", linewidth=2, alpha=0.4)
# class AnyObjectHandler(HandlerBase):
#     def create_artists(self, legend, orig_handle,
#                        x0, y0, width, height, fontsize, trans):
#         l1 = plt.Line2D([x0,y0+width], [0.7*height,0.7*height],
#                                                 linestyle='--', color=colors[r*2])
#         l2 = plt.Line2D([x0,y0+width], [0.3*height,0.3*height], color=colors[r*2+1])
#         return [l1, l2]
# plt.legend([object], [f'Object ratio : {ratios[0]:.2f}'],  handler_map={object: AnyObjectHandler()})
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Test Accuracy")
axs[0].set_ylim(60, 80)
# plt.tight_layout()
# plt.savefig("figure/loss_acc.pdf")

for r in range(len(ratios)):
    axs[1].plot(loss[r], label=f"ratio: {ratios[r]}", color=colors[r], alpha=0.7, linewidth=4)
    axs[1].plot(smooth_data(loss[r], window_size=10), label=f"ratio: {ratios[r]}", color=colors_darker[r], alpha=0.8, linewidth=6)
    axs[1].set_xlabel("Step")
    axs[1].set_ylabel("Loss")
# axs[0].set_xlim(0, 15000)
axs[1].set_yscale('log')
axs[0].legend(ncols=2, loc='lower right')

plt.tight_layout()
plt.savefig(f'figure/{method}.pdf', format='pdf')
plt.show()

In [ ]:
from PIL import Image
img1 = Image.open('/home/dataset/waterbird_single_background/0.05/001.Black_footed_Albatross/Black_Footed_Albatross_0001_796111.jpg')
img2 = Image.open('/home/dataset/waterbird_single_background/0.05/016.Painted_Bunting/Painted_Bunting_0002_16887.jpg')
img3 = Image.open('/home/dataset/waterbird_single_background/0.2/001.Black_footed_Albatross/Black_Footed_Albatross_0001_796111.jpg')
img4 = Image.open('/home/dataset/waterbird_single_background/0.2/030.Fish_Crow/Fish_Crow_0001_26031.jpg')

fig, axes = plt.subplots(2, 2, figsize=(7.5, 7.5))
plt.subplots_adjust(wspace=0.01)

axes[0, 0].imshow(img1)
axes[0, 0].axis('off')
axes[0, 0].set_title('Object ratio : 0.05', y=-0.13)

axes[0, 1].imshow(img2)
axes[0, 1].axis('off')
axes[0, 1].set_title('Object ratio : 0.05', y=-0.13)

axes[1, 0].imshow(img3)
axes[1, 0].axis('off')
axes[1, 0].set_title('Object ratio : 0.3', y=-0.13)

axes[1, 1].imshow(img4)
axes[1, 1].axis('off')
axes[1, 1].set_title('Object ratio : 0.3', y=-0.13)

plt.savefig('figure/waterbirds_size_compare.pdf', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
import torchvision.transforms.v2 as transforms
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from PIL.Image import Image
from PIL import Image, ImageOps, ImageFilter
import pandas as pd
import torchvision
import cv2

device = torch.device("cuda")
size = 128


class WaterbirdDataset(Dataset):
    def __init__(self, root_dir, metadata_path, split='train', transform=None, balance_classes=True, no_aug=False):
        self.root_dir = root_dir
        self.transform = transform
        self.split = split
        self.no_aug = no_aug
        self.metadata = pd.read_csv(metadata_path)

        split_dict = {'train': 0, 'val': 1, 'test': 2}

        self.metadata = self.metadata[self.metadata['split'] == split_dict[split]]
        if balance_classes:
            landbird_data = self.metadata[self.metadata['y'] == 0]
            waterbird_data = self.metadata[self.metadata['y'] == 1]

            min_class_size = min(len(landbird_data), len(waterbird_data))
            # min_class_size = 256

            landbird_balanced = landbird_data.sample(min_class_size, random_state=1)
            waterbird_balanced = waterbird_data.sample(min_class_size, random_state=1)

            self.metadata = pd.concat([landbird_balanced, waterbird_balanced])

        self.metadata = self.metadata.reset_index(drop=True)

        self.size_transform = transforms.Resize(size)

        self.labels = []
        self.places = []
        self.img_ids = []
        self.img = []
        mean = 0.0
        var = 0.0

        for idx in range(len(self.metadata)):
            img_filename = self.metadata.loc[idx, 'img_filename']
            self.labels.append(int(self.metadata.loc[idx, 'y']))
            self.places.append(int(self.metadata.loc[idx, 'place']))
            self.img_ids.append(self.metadata.loc[idx, 'img_id'] if 'img_id' in self.metadata.columns else idx)
            image = Image.open(os.path.join(self.root_dir, img_filename)).convert('RGB')
            image_resize = self.size_transform(image)
            self.img.append(image_resize)
            image_preprocess = transforms.ToTensor()(image_resize)
            mean += image_preprocess.mean(dim=[1, 2])
            var += image_preprocess.var(dim=[1, 2])

        mean /= len(self.img)
        var /= len(self.img)
        std = torch.sqrt(var)
        print(mean, std)

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        self.transform_image = transforms.ToTensor()

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        image_original = self.img[idx]
        label = self.labels[idx]
        places = self.places[idx]
        if self.split == 'test':
            return self.transform(image_original), self.transform_image(image_original), label, places
        return self.transform(image_original), label, places

In [ ]:
class BarlowTwins(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torchvision.models.resnet34(zero_init_residual=True, weights=None)
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.size_in = self.backbone.fc.in_features
        self.size_out = 256
        self.backbone.fc = nn.Identity()

        sizes = [self.size_in, self.size_out, self.size_out]
        layers = []
        for i in range(len(sizes) - 2):
            layers.append(nn.Linear(sizes[i], sizes[i + 1], bias=False))
            layers.append(nn.BatchNorm1d(sizes[i + 1]))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Linear(sizes[-2], sizes[-1], bias=False))
        self.projector = nn.Sequential(*layers)

    def forward(self, x, use_projector=True):
        if not use_projector:
            return self.backbone(x)
        z = self.projector(self.backbone(x))
        return z

In [ ]:
def generate_resnet_from_dict(path):
    model = BarlowTwins()
    model.load_state_dict(torch.load(path))
    return model.backbone, model.size_in

In [ ]:
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

image_map = {}
pt_list = range(40, 40+80*8+1, 120)
count_ocean = 10
count_forest = 10
for ratio in ratios:
    root_dir = f"/home/dataset/waterbird_single_background/{ratio}"
    metadata_path = f"/home/dataset/waterbird_single_background/{ratio}/metadata.csv"

    test_dataset = WaterbirdDataset(
        root_dir=root_dir,
        metadata_path=metadata_path,
        split='test',
        balance_classes=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
    )

    image_map[ratio] = []
    save_path = f"{directory_models}/{ratio}/linear"
    for i in range(len(pt_list)):
        print(f"Processing {pt_list[i]}")
        model, size_out = generate_resnet_from_dict(f"{directory_models}/{ratio}/{pt_list[i]}.pt")
        model = model.to(device)
        linear = nn.Linear(size_out, 2)
        linear.load_state_dict(torch.load(f"{directory_models}/{ratio}/linear/{pt_list[i]}.pt"))
        linear = linear.to(device)

        # target_layers = [model.layer1[-1], model.layer2[-1], model.layer3[-1], model.layer4[-1]]
        target_layers = [model.layer4[-1]]
        model_grad = nn.Sequential(model, linear).to(device)

        i_ocean = 0
        i_forest = 0
        image_org = []
        image_grad = []
        for index, (image_tensor, image, label, background) in enumerate(test_loader):
            if label == background:
                continue
            if i_ocean >= count_ocean and i_forest >= count_forest:
                break
            if label == 0 and i_ocean >= count_ocean:
                continue
            if label == 1 and i_forest >= count_forest:
                continue
            # targets = [ClassifierOutputTarget(1)]
            targets = None
            current_image = image[0, :, :, :]
            current_image = current_image.permute(1, 2, 0)
            current_image = current_image.cpu().numpy()
            with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
                grayscale_cam = cam(input_tensor=image_tensor, targets=targets)
                grayscale_cam = grayscale_cam[0, :]
                visualization = show_cam_on_image(current_image, grayscale_cam, use_rgb=True)
                model_outputs = cam.outputs
            # plt.imshow(to_pil_image(current_image))
            # plt.show()
            # plt.imshow(to_pil_image(visualization))
            # plt.show()
            image_org.append(to_pil_image(current_image))
            image_grad.append(to_pil_image(visualization))
            # print(f"correct answer:{label.item()}, spurious answer: {background.item()}")
            # print(f"model: {model_outputs.detach().cpu()}")
            if label == 0:
                i_ocean += 1
            else:
                i_forest += 1

        if len(image_map[ratio]) == 0:
            image_map[ratio].append(image_org)
        image_map[ratio].append(image_grad)

In [ ]:
n_images = len(pt_list) - 1
print(len(ratios)*2, n_images)
scale = 0.75
fig, axes = plt.subplots(n_images, len(ratios)*2, figsize=(len(ratios)*2*scale, n_images*scale), sharey=True, sharex=True)
target_ocean = 7
target_forest = count_ocean + 0
for ratio_index in range(len(ratios)):
    current_images = image_map[ratios[ratio_index]]
    for j in range(n_images):
        axes[j, ratio_index].imshow(current_images[j][target_ocean])
        axes[j, ratio_index+len(ratios)].imshow(current_images[j][target_forest])
        axes[j, ratio_index+len(ratios)].axis('off')

        if ratio_index == 0:
            axes[j, 0].axes.get_xaxis().set_ticks([])
            axes[j, 0].axes.get_yaxis().set_ticks([])
            axes[j, 0].set_ylabel(f'Step {pt_list[j]}', rotation=0, labelpad=22, va='center', fontsize=8)
        else:
            axes[j, ratio_index].axis('off')

        if j == 0:
            axes[0, ratio_index].set_title(f'Ratio {ratios[ratio_index]:.3f}\nOcean', fontsize=8)
            axes[0, ratio_index+len(ratios)].set_title(f'Ratio {ratios[ratio_index]:.3f}\nForest', fontsize=8)

plt.subplots_adjust(wspace=0.02, hspace=0.02)
plt.tight_layout(pad=0.1)
plt.savefig("figure/gradcam.pdf")
plt.show()